1) Accessing data for classification

In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from data_aggregation.creating_df_full import build_full_dataset
import json


In [ ]:
X, y, df_full = build_full_dataset(last_season=2023, n_seasons=7, return_full=True)


In [7]:
df_full.to_csv("nba_dataset_2017_2023.csv", index=False)



2) Model Creating

In [8]:


# 1. Załaduj dane z sezonów 2017–2023
df_full = pd.read_csv("nba_dataset_2017_2023.csv")
X = df_full.drop(columns=["Player", "Team", "Pos", "season", "target"])
y = df_full["target"]

# 2. Podziel dane: trening (2017–2022), predykcja (2023)
mask_train = df_full["season"] < 2023
mask_pred = df_full["season"] == 2023

X_train = X[mask_train]
y_train = y[mask_train]
X_pred = X[mask_pred]
df_pred_meta = df_full[mask_pred][["Player", "is_rookie"]].reset_index(drop=True)

# 3. Trenuj model
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

# 4. Predykcja — prawdopodobieństwa klas
probas = model.predict_proba(X_pred)
df_pred = pd.DataFrame(probas, columns=model.classes_)
df_pred["Player"] = df_pred_meta["Player"]
df_pred["is_rookie"] = df_pred_meta["is_rookie"]

# 5. Tworzenie piątek

results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# 6. Zapis do JSON
with open("classification_result.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Gotowe")

Gotowe
